# Left-Wing Non-Fiction Books — Goodreads Scraper
Scrapes Goodreads author pages to collect books by a curated author list.
Two-step per author: (1) search for author ID, (2) paginate through their book list.
Saves to `leftpolitics_raw.csv` with the same schema as the Open Library version.

In [1]:
import subprocess
import pandas as pd
import time
import re
from bs4 import BeautifulSoup
from urllib.parse import urlencode

DELAY = 3.0

CURL_CMD = [
    "curl", "-s", "-L", "--max-time", "20",
    "-A", "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "-H", "Accept-Language: en-US,en;q=0.9",
    "-H", "Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
]

def curl_get(url, params=None):
    if params:
        url = f"{url}?{urlencode(params)}"
    result = subprocess.run(CURL_CMD + [url], capture_output=True, text=True, timeout=25)
    return result.stdout

In [2]:
AUTHORS = [
    "Angela Davis",
    "James Baldwin",
    "Mahmoud Darwish",
    "Noam Chomsky",
    "Naomi Klein",
    "bell hooks",
    "Howard Zinn",
    "Eduardo Galeano",
    "Frantz Fanon",
    "Rosa Luxemburg",
    "Antonio Gramsci",
    "Michel Foucault",
    "Cornel West",
    "Ta-Nehisi Coates",
    "Robin D.G. Kelley",
    "W.E.B. Du Bois",
    "Audre Lorde",
    "Malcolm X",
    "Cedric Robinson",
    "Saidiya Hartman",
    "Keeanga-Yamahtta Taylor",
    "Patricia Hill Collins",
    "Manning Marable",
    "Grace Lee Boggs",
    "Ibram X. Kendi",
    "C.L.R. James",
    "Aimé Césaire",
    "Walter Rodney",
    "Achille Mbembe",
    "Stuart Hall",
    "Paulo Freire",
    "Vijay Prashad",
    "José Carlos Mariátegui",
    "Sara Ahmed",
    "Silvia Federici",
    "Gayatri Chakravorty Spivak",
    "David Graeber",
    "Mark Fisher",
    "Slavoj Žižek",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}
DELAY = 3.0

In [3]:
# --- DEBUG: run this alone first to verify connectivity and selectors ---
html = curl_get("https://www.goodreads.com/search", {"q": "Angela Davis", "search_type": "authors", "page": "1"})
print("HTML length:", len(html))

soup = BeautifulSoup(html, "html.parser")
print("Title:", soup.title.get_text(strip=True) if soup.title else "NONE - bad response")

rows = soup.select("tr[itemtype='http://schema.org/Book']")
print(f"Book rows: {len(rows)}")

if rows:
    r = rows[0]
    title  = r.select_one("a.bookTitle span[itemprop='name']")
    author = r.select_one("a.authorName span[itemprop='name']")
    grey   = r.select_one("span.greyText.smallText")
    year_m = re.search(r'published\s+(\d{4})', grey.get_text() if grey else "")
    ed_tag = next((a for a in r.select("a.greyText") if "edition" in a.get_text()), None)
    print("\n--- First row ---")
    print("Title:   ", title.get_text(strip=True)  if title  else "NOT FOUND")
    print("Author:  ", author.get_text(strip=True) if author else "NOT FOUND")
    print("Year:    ", year_m.group(1) if year_m else "NOT FOUND")
    print("Editions:", ed_tag.get_text(strip=True) if ed_tag else "NOT FOUND")

HTML length: 189203
Title: Search results for "Angela Davis" (showing 1-20 of 796 books)
Book rows: 20

--- First row ---
Title:    An Autobiography
Author:   Angela Y. Davis
Year:     1974
Editions: 67 editions


In [4]:
def parse_page(html, queried_author):
    soup = BeautifulSoup(html, "html.parser")
    records = []

    for row in soup.select("tr[itemtype='http://schema.org/Book']"):
        title_tag = row.select_one("a.bookTitle span[itemprop='name']")
        if not title_tag:
            continue
        title = title_tag.get_text(strip=True)

        book_link = row.select_one("a.bookTitle")
        book_href = book_link["href"].split("?")[0] if book_link else ""

        author_tag = row.select_one("a.authorName span[itemprop='name']")
        author = author_tag.get_text(strip=True) if author_tag else queried_author

        grey = row.select_one("span.greyText.smallText")
        grey_text = grey.get_text() if grey else ""

        year = None
        m = re.search(r'published\s+(\d{4})', grey_text)
        if m:
            year = int(m.group(1))

        edition_count = 0
        ed_tag = next((a for a in row.select("a.greyText") if "edition" in a.get_text()), None)
        if ed_tag:
            m2 = re.search(r'[\d,]+', ed_tag.get_text())
            if m2:
                edition_count = int(m2.group().replace(",", ""))

        records.append({
            "title": title,
            "author": author,
            "year_published": year,
            "subjects": "",
            "edition_count": edition_count,
            "open_library_key": book_href,
            "queried_author": queried_author,
        })
    return records, soup


def scrape_author(author_name, max_pages=10):
    all_records = []
    for page in range(1, max_pages + 1):
        html = curl_get(
            "https://www.goodreads.com/search",
            {"q": author_name, "search_type": "authors", "page": page},
        )
        records, soup = parse_page(html, author_name)
        if not records:
            break
        all_records.extend(records)
        if not soup.select_one("a[rel='next']") and not soup.select_one(".next_page"):
            break
        time.sleep(DELAY)
    return all_records

In [5]:
all_records = []

for author in AUTHORS:
    print(f"Scraping: {author}...")
    try:
        books = scrape_author(author)
        print(f"  → {len(books)} books found")
        all_records.extend(books)
    except Exception as e:
        print(f"  Error: {e}")
    time.sleep(DELAY)

print(f"\nTotal records collected: {len(all_records)}")

Scraping: Angela Davis...
  → 159 books found
Scraping: James Baldwin...
  → 200 books found
Scraping: Mahmoud Darwish...
  → 194 books found
Scraping: Noam Chomsky...
  → 199 books found
Scraping: Naomi Klein...
  → 71 books found
Scraping: bell hooks...
  → 196 books found
Scraping: Howard Zinn...
  → 0 books found
Scraping: Eduardo Galeano...
  → 192 books found
Scraping: Frantz Fanon...
  → 199 books found
Scraping: Rosa Luxemburg...
  → 198 books found
Scraping: Antonio Gramsci...
  → 199 books found
Scraping: Michel Foucault...
  → 200 books found
Scraping: Cornel West...
  → 192 books found
Scraping: Ta-Nehisi Coates...
  → 198 books found
Scraping: Robin D.G. Kelley...
  → 107 books found
Scraping: W.E.B. Du Bois...
  → 200 books found
Scraping: Audre Lorde...
  → 195 books found
Scraping: Malcolm X...
  → 200 books found
Scraping: Cedric Robinson...
  → 53 books found
Scraping: Saidiya Hartman...
  → 38 books found
Scraping: Keeanga-Yamahtta Taylor...
  → 23 books found
Scrapi

In [6]:
df = pd.DataFrame(all_records)

df = df.drop_duplicates(subset=["title", "author"])
df = df.sort_values(["queried_author", "year_published"], na_position="last")
df = df.reset_index(drop=True)

print(f"Unique records after deduplication: {len(df)}")
df.head(10)

Unique records after deduplication: 5187


,title,author,year_published,subjects,edition_count,open_library_key,queried_author
0,La naissance du maquis dans le Sud-Cameroun (H...,Achille Mbembe,1996.0,,2,/book/show/74573817-la-naissance-du-maquis-dan...,Achille Mbembe
1,On the Postcolony,Achille Mbembe,2001.0,,17,/book/show/149757.On_the_Postcolony,Achille Mbembe
2,Johannesburg: The Elusive Metropolis (Volume 16),Sarah Nuttall,2004.0,,5,/book/show/327816.Johannesburg,Achille Mbembe
3,"África Insubmissa: Cristianismo, poder e estad...",Achille Mbembe,2005.0,,2,/book/show/45914660-frica-insubmissa,Achille Mbembe
4,Je est un autre - Pour une identité-monde,Kebir-Mustapha Ammi,2010.0,,3,/book/show/44436174-je-est-un-autre---pour-une...,Achille Mbembe
5,Necropolitics,Achille Mbembe,2016.0,,24,/book/show/44901216-necropolitics,Achille Mbembe
6,Politiques de l'inimitié,Achille Mbembe,2016.0,,2,/book/show/30246397-politiques-de-l-inimiti,Achille Mbembe
7,Brutalisme,Achille Mbembe,2020.0,,18,/book/show/51131710-brutalisme,Achille Mbembe
8,The Earthly Community: Reflections on the Last...,Achille Mbembe,2023.0,,15,/book/show/211097016-the-earthly-community,Achille Mbembe
9,Critique of Black Reason,Achille Mbembe,NaN,,4,/book/show/30757980-critique-of-black-reason,Achille Mbembe


In [7]:
output_path = "leftpolitics_raw.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

Saved 5187 rows to leftpolitics_raw.csv
